> **Provenance only — not runnable from the public repository; outputs intentionally cleared.**
>
> This notebook documents the raw → cleaned → analysis-ready → **blinded** cleaning
> pipeline. It reads the **non-public** source file `data/BGA_merged_all_20260208.csv`
> (original `BGA_###` study IDs, `Education`, `HandPref`, and other fields removed
> during de-identification), which is **not distributed** with this repository.
>
> Its cell outputs are **deliberately left empty**: executed previews would display
> row-level, non-blinded cohort data (original IDs and dropped fields) and could
> expose the `subj_### ↔ BGA_###` mapping, defeating the de-identification. Do **not**
> commit executed outputs of this notebook.
>
> The only public, runnable analysis path is
> **`03_table_1_generation_blinded.ipynb`**, which uses the released blinded dataset.

# 01 - Cohort Data Cleaning (raw to cleaned to analysis to blinded)

Last updated: A.L. 2026-06-28

This notebook **documents and reproduces** the data-cleaning procedure that turns the single raw cohort export into the three derived files consumed by the rule-based analysis pipeline (`scripts/neuropsych_pipeline.py`). Its purpose is to make every cleaning decision transparent and reproducible for review.

## Lineage

| Stage | Output file | Shape | What happens |
|---|---|---|---|
| raw | `data/BGA_merged_all_20260208.csv` | 105 x 74 | single hand-merged export (see `data/_sources/`) |
| 1 - cleaning | `..._cleaned.csv` | 105 x 82 | technical cleaning + 8 QC flags |
| 2 - analysis extract | `..._cleaned_for_analysis.csv` | 105 x 68 | drop RBANS indices & flags, derive `RBANS_Sum_Raw` |
| 3 - blinded release | `..._cleaned_for_analysis_blinded.csv` | 105 x 64 | drop identifiers, relabel subjects -> **only public file** |

## Single source of truth

To prevent documentation and code from drifting apart, the exact transforms live in **`scripts/clean_cohort_data.py`**. This notebook *imports* that module, prints the exact source of each stage with `inspect.getsource`, runs it on the raw data, and verifies the outputs cell-for-cell against the committed files. The narrative (rationale, definitions) lives here; the canonical implementation is the script.

> **Data access.** The raw, `cleaned`, and `for_analysis` files contain non-blinded data and are **not** part of the public repository, so this notebook runs only in the full-data environment. The downstream `03_table_1_generation_blinded.ipynb` runs on the public blinded file alone. The authoritative rule list is `data/README_cleaning.md`; raw-file provenance is `data/_sources/README_provenance.md`.

In [ ]:
import importlib
import inspect
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def resolve_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "scripts" / "clean_cohort_data.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate scripts/clean_cohort_data.py")


ROOT = resolve_repo_root()
sys.path.insert(0, str(ROOT / "scripts"))
import clean_cohort_data as ccd  # canonical cleaning implementation
importlib.reload(ccd)

RAW_PATH = ccd._DATA / ccd.RAW_FILE
raw = ccd.load_raw(RAW_PATH)
print("Repo root:", ROOT)
print("Raw file :", RAW_PATH)
print("Raw shape:", raw.shape, " Group:", dict(raw["Group"].value_counts()))

## Stage 1 - Technical cleaning + QC flags  ->  `..._cleaned.csv`

The raw file is read **as strings** with blanks preserved (`dtype=str, keep_default_na=False`) so nothing is silently coerced on load. Stage 1 then makes only minimal, meaning-preserving changes and records data-quality issues as explicit flags rather than editing values.

**Cleaning operations**
- Blank / whitespace-only strings -> empty (explicit missing).
- `Education`: text-like integers coerced to numeric (`"12"` -> `"12.0"`).
- `HandPref`: the atypical label `"Right+ left"` normalised to `"Ambidextrous"`.

**Eight QC flags** (appended as `True`/`False` columns)
- `flag_missing_bis_block` / `..._fatigue_block` / `..._hads_block` / `..._cpt_block` / `..._rbans_block` - the *entire* instrument block is blank for that subject.
- `flag_hc_with_ibs_sss` - a healthy control nonetheless carries an IBS severity score.
- `flag_rbans_sum_mismatch` - recorded `RBANS_Sum_Index` differs from the sum of the 5 RBANS domain indices (reproduces the two subjects the original notebook hardcoded: `subj_089`, `subj_101`).
- `flag_tfs_vs_13_item_sum_mismatch` - recorded Chalder total differs from the 13-item FSS sum (fires when the two extra FSS items beyond the canonical 11-item Chalder scale are endorsed).

The flags are **informational**: recorded values are never overwritten. Downstream analyses simply exclude flagged values from the specific affected comparisons (see `data/README_cleaning.md`).

In [ ]:
print(inspect.getsource(ccd.build_cleaned))
print(inspect.getsource(ccd.compute_flags))

In [ ]:
cleaned = ccd.build_cleaned(raw)
print("cleaned shape:", cleaned.shape)

flag_summary = pd.DataFrame({"n_flagged": cleaned[ccd.FLAG_COLUMNS].eq("True").sum()})
flag_summary["pct"] = (flag_summary["n_flagged"] / len(cleaned) * 100).round(1)
display(flag_summary)
display(cleaned[["Subject", "Group", "Education", "HandPref"]].head())

### The two value-discrepancy flags, in context

Two flags mark genuine number discrepancies (the others are structural missingness):

- **`flag_tfs_vs_13_item_sum_mismatch` (~50 subjects).** The Chalder Fatigue Scale total is canonically the sum of items 1-11; the export also stores two additional FSS items (12-13). Wherever those are endorsed, the recorded total differs from a naive 13-item sum. This is expected and does not indicate an error in the recorded Chalder total used downstream.
- **`flag_rbans_sum_mismatch` (2 subjects).** For `subj_089` and `subj_101` the recorded RBANS composite does not equal the sum of the five domain indices. These two rows are flagged and excluded from the RBANS-sum analysis only.

Listing the flagged subjects:

In [ ]:
mask = cleaned["flag_rbans_sum_mismatch"].eq("True") | cleaned["flag_hc_with_ibs_sss"].eq("True")
display(
    cleaned.loc[mask, ["Subject", "Group", "RBANS_Sum_Index", "IBS_SSS",
                       "flag_rbans_sum_mismatch", "flag_hc_with_ibs_sss"]]
)
print("Chalder vs 13-item mismatches:",
      int(cleaned["flag_tfs_vs_13_item_sum_mismatch"].eq("True").sum()))

## Stage 2 - Analysis extract  ->  `..._cleaned_for_analysis.csv`

The cleaned file is slimmed to exactly the columns the deterministic pipeline reads (105 x 68):

- **Drop** the seven RBANS summary/index columns (the pipeline derives its own cohort-relative summaries) and the eight `flag_*` columns (QC metadata, not model inputs).
- **Derive** `RBANS_Sum_Raw` = row-sum of the 12 RBANS raw subtests (blank when the whole RBANS block is missing).
- **Reorder** so the RBANS subtests follow the analysis order (two recalls, then two recognitions) and `RBANS_Sum_Raw` sits immediately before `TFS_Chalder`.

In [ ]:
print(inspect.getsource(ccd.build_for_analysis))
analysis = ccd.build_for_analysis(cleaned)
print("for_analysis shape:", analysis.shape)
print("dropped vs cleaned :", sorted(set(cleaned.columns) - set(analysis.columns)))
print("added   vs cleaned :", sorted(set(analysis.columns) - set(cleaned.columns)))
display(analysis[["Subject", "Group", "RBANS_Sum_Raw", "TFS_Chalder"]].head())

## Stage 3 - Blinded release  ->  `..._cleaned_for_analysis_blinded.csv`

De-identification for public release (105 x 64), the only file in the public repo:

- **Drop** four free-text / indirect-identifier columns: `Education`, `HandPref`, `Mothertounge`, `TestAdmin`.
- **Relabel** `Subject` to sequential anonymous IDs `subj_001 ... subj_105` by row order (no link back to the clinical `BGA_xxx` IDs).

In [ ]:
print(inspect.getsource(ccd.build_blinded))
blinded = ccd.build_blinded(analysis)
print("blinded shape:", blinded.shape)
print("dropped vs analysis:", sorted(set(analysis.columns) - set(blinded.columns)))
display(blinded[["Subject", "Gender", "Group", "TFS_Chalder"]].head())

## Verification - exact value-level reproduction

Each generated frame is compared cell-for-cell against the committed file. Comparison is value-level (numeric where both parse as numbers, otherwise stripped string) because the committed `..._cleaned.csv` is stored in a cosmetically space-padded `"; "` layout, whereas the canonical writer emits plain `";"`.

In [ ]:
ok = True
ok &= ccd.verify(cleaned,  ccd._DATA / ccd.CLEANED_FILE,  "cleaned")
ok &= ccd.verify(analysis, ccd._DATA / ccd.ANALYSIS_FILE, "for_analysis")
ok &= ccd.verify(blinded,  ccd._DATA / ccd.BLINDED_FILE,  "blinded")
print("\n" + ("ALL FILES REPRODUCED EXACTLY (value-level)." if ok else "VERIFICATION FAILED."))
assert ok, "value-level verification failed"

## Regenerate from the command line

This notebook is the narrated companion to the script; the identical result is produced headlessly by:

```bash
python scripts/clean_cohort_data.py                  # write to data/reproduced/ + verify
python scripts/clean_cohort_data.py --write-inplace  # overwrite data/*.csv
```

### Provenance & legacy
- Stage-1 logic reproduces the original cleaning cell of `01_data_exploration_20260311.ipynb`.
- An earlier `..._analysis_ready.csv` with `use_*` mask columns is **superseded** and no longer produced; the pipeline reads `..._cleaned_for_analysis.csv` (see `notebooks/04_cleaned_data_results.ipynb`).
- Upstream, the raw file itself is reproducible via `data/_sources/build_raw_cohort.py`.